# Conversion Profile 기반 FHIR 연구 검증

연구계획 수정 메모의 환자 분할·고정 추출·Profile 생성/보정·Direct/Hybrid 대응 평가를 실행합니다. 각 RUN 플래그를 켜고 원천 데이터 경로를 지정하세요.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").is_file())
CONFIG = ROOT / "experiments/configs/pilot.json"
RUN_DIR = ROOT / "artifacts/notebook-research"
DATASET = "mimic_iv"
SPEC = ROOT / "research/fhir_profile/data/specs" / f"{DATASET}.json"
PATIENT_KEYS = ROOT / "data" / f"{DATASET}-patient-keys.json"
SPLIT = RUN_DIR / "split.json"
PREPARATION_ROLE = "development" if DATASET == "mimic_iv" else "calibration"
PREPARATION = RUN_DIR / "preparation.json"
TEST_COHORT = RUN_DIR / "test.json"
INDEX = RUN_DIR / "fhir-index.json"
LOCK = RUN_DIR / "policy.lock.json"
PROFILES = RUN_DIR / "profiles.json"
REFERENCE = ROOT / "data/reference" / DATASET
REFERENCE_PROFILES = None
CSV_DIR = None
DB_URL_ENV = "SOURCE_DB_URL"
RESULTS = RUN_DIR / "results"
RUN_SPLIT = False
RUN_PREPARE = False
RUN_BUILD_RAG = False
RUN_FREEZE = False
RUN_GENERATE_PROFILE = False
RUN_EVALUATE = False
RUN_COMPARE_RAG = False
LEFT_MANIFEST = RUN_DIR / "no-rag/results/manifest.json"
RIGHT_MANIFEST = RUN_DIR / "rag/results/manifest.json"

def research(*arguments):
    command = [sys.executable, str(ROOT / "main.py"), "research", *map(str, arguments)]
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)

config = json.loads(CONFIG.read_text())
index_args = ["--index", INDEX] if config["rag"].get("enabled", True) else []
display({"dataset": DATASET, "model": config["model"], "rag": config["rag"],
         "validator": config["validation"], "output_directory": str(RUN_DIR)})


## 환자 분할과 원천 추출

동일 환자의 모든 입원·이벤트를 같은 분할에 포함합니다. 추출 COUNT와 실제 행 수를 확인합니다.

In [ ]:
if RUN_SPLIT:
    research("split", "--patients", PATIENT_KEYS, "--dataset", DATASET, "--output", SPLIT)
if RUN_PREPARE:
    source_args = ["--csv-dir", CSV_DIR] if CSV_DIR else ["--db-url-env", DB_URL_ENV]
    for role, destination in [(PREPARATION_ROLE, PREPARATION), ("test", TEST_COHORT)]:
        research("prepare", "--spec", SPEC, "--split", SPLIT, "--role", role,
                 *source_args, "--output", destination)
if TEST_COHORT.is_file():
    cohort = json.loads(TEST_COHORT.read_text())
    display({kind: {"records": len(batch["records"]), "extraction": batch["extraction"]}
             for kind, batch in cohort["resources"].items()})


## RAG·정책 고정과 Profile 생성

모델·프롬프트·코드·스키마·Validator·source_checks를 고정합니다. 보정은 준비 환자의 원천 보존 검사와 표준 검증 결과를 사용합니다.

In [ ]:
if RUN_BUILD_RAG:
    research("build-rag", "--config", CONFIG, "--output", INDEX)
if RUN_FREEZE:
    research("freeze", "--config", CONFIG, *index_args, "--output", LOCK)
if RUN_GENERATE_PROFILE:
    research("generate-profile", "--lock", LOCK, *index_args,
             "--records", PREPARATION, "--output", PROFILES)
if PROFILES.is_file():
    profiles = json.loads(PROFILES.read_text())
    display({kind: {"status": entry.get("status"), "metrics": entry.get("calibration_metrics"),
                    "history": entry.get("history")} for kind, entry in profiles["profiles"].items()})


## Direct·Hybrid·Self-Correction 평가

독립 참조와 매핑 정답은 평가 단계에서 읽습니다. identifier 정렬 규칙은 evaluation.reference_by_dataset에 지정합니다.

In [ ]:
if RUN_EVALUATE:
    reference_args = ["--reference", REFERENCE]
    if REFERENCE_PROFILES:
        reference_args += ["--reference-profiles", REFERENCE_PROFILES]
    research("evaluate", "--lock", LOCK, *index_args, "--records", TEST_COHORT,
             "--profiles", PROFILES, *reference_args, "--groups", "B1", "P1", "P2",
             "--output-dir", RESULTS)
manifest_path = RESULTS / "manifest.json"
if manifest_path.is_file():
    result = json.loads(manifest_path.read_text())
    display({"patient_count": result["patient_count"], "correction": result["correction_summary"]})
    display({group: {"fhir_bundle_pass_rate": data["fhir_bundle_pass_rate"],
                      "resource_count_alignment": data["resource_count_alignment"],
                      "mapping_accuracy": data["mapping_accuracy"], "patients": data["patients"]}
             for group, data in result["groups"].items()})
    display(result["paired_comparisons"])
    display(result["cost_summary"])
    display(result["pilot_summary"])
else:
    display({"result": "실험 결과 파일을 준비하세요.", "path": str(manifest_path)})


## RAG 유무 대응 비교

같은 환자·cohort·독립 참조·모델·코드에서 rag.enabled만 변경된 실험을 비교합니다. Bootstrap 재표집 단위는 환자이며, 환자 1명인 경우 신뢰구간은 미평가입니다. Pilot 표본수는 대응 차이의 CI 정밀도 기반 근사치로 기록합니다.

In [ ]:
if RUN_COMPARE_RAG:
    research("compare-results", "--left", LEFT_MANIFEST, "--right", RIGHT_MANIFEST,
             "--rag-ablation", "--output", RUN_DIR / "rag-comparison.json")
